In [1]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

DB_PATH = Path("../music_artist_insights.duckdb")

con = duckdb.connect(
    str(DB_PATH),
    read_only=True
)

print("Connected to DuckDB")

Connected to DuckDB


In [2]:
release_yearly = con.execute("""
    SELECT
        artist_name,
        release_year,
        total_releases,
        core_releases,
        singles,
        albums,
        eps,
        remixes,
        compilations,
        dj_mixes,
        live_releases,
        comparable_ytd_releases,
        comparable_ytd_core_releases,
        dataset_through_date
    FROM gold_release_metrics
    ORDER BY artist_name, release_year
""").fetchdf()

release_yearly

,artist_name,release_year,total_releases,core_releases,singles,albums,eps,remixes,compilations,dj_mixes,live_releases,comparable_ytd_releases,comparable_ytd_core_releases,dataset_through_date
0,Armin van Buuren,1996,3,2,3,0,0,1,0,0,0,3,2,2026-09-25
1,Armin van Buuren,1999,8,4,4,2,2,2,2,2,0,7,4,2026-09-25
2,Armin van Buuren,2000,5,3,3,2,0,0,2,2,0,5,3,2026-09-25
3,Armin van Buuren,2001,5,3,3,2,0,0,2,2,0,5,3,2026-09-25
4,Armin van Buuren,2002,3,1,2,1,0,1,1,1,0,3,1,2026-09-25
5,Armin van Buuren,2003,7,6,2,2,3,0,1,1,1,5,5,2026-09-25
6,Armin van Buuren,2004,8,3,2,5,1,0,5,4,1,7,3,2026-09-25
7,Armin van Buuren,2005,11,8,4,4,3,0,3,2,0,8,6,2026-09-25
8,Armin van Buuren,2006,10,6,5,3,2,0,4,2,0,5,3,2026-09-25
9,Armin van Buuren,2007,9,4,4,5,0,0,5,3,1,7,4,2026-09-25


In [3]:
artist_baselines = (
    release_yearly[
        release_yearly["release_year"] < 2026
    ]
    .groupby("artist_name")
    .agg(
        historical_avg_releases=("total_releases", "mean"),
        historical_median_releases=("total_releases", "median"),
        historical_avg_core_releases=("core_releases", "mean"),
        active_years=("release_year", "nunique"),
    )
    .reset_index()
)

artist_baselines.round(1)

,artist_name,historical_avg_releases,historical_median_releases,historical_avg_core_releases,active_years
0,Armin van Buuren,19.1,20.5,9.6,28
1,Tiësto,10.6,10.0,6.1,30


In [4]:
current = (
    release_yearly[
        release_yearly["release_year"] == 2026
    ][
        [
            "artist_name",
            "comparable_ytd_releases",
            "comparable_ytd_core_releases",
            "dataset_through_date",
        ]
    ]
    .copy()
)

context = current.merge(
    artist_baselines,
    on="artist_name",
    how="left"
)

context["ytd_vs_historical_avg_pct"] = (
    (
        context["comparable_ytd_releases"]
        / context["historical_avg_releases"]
    )
    - 1
) * 100

context.round(1)

/var/folders/n3/m2px247567bd0zrky2q6tmlw0000gn/T/ipykernel_94726/3765762596.py:29: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  context.round(1)


,artist_name,comparable_ytd_releases,comparable_ytd_core_releases,dataset_through_date,historical_avg_releases,historical_median_releases,historical_avg_core_releases,active_years,ytd_vs_historical_avg_pct
0,Armin van Buuren,28,18,2026-09-25,19.1,20.5,9.6,28,46.5
1,Tiësto,3,3,2026-07-17,10.6,10.0,6.1,30,-71.6


In [5]:
catalogue = con.execute("""
    SELECT
        artist_name,
        first_release_date,
        primary_type,
        is_remix,
        is_compilation,
        is_live,
        is_dj_mix
    FROM release_catalogue
""").fetchdf()

catalogue["first_release_date"] = pd.to_datetime(
    catalogue["first_release_date"]
)

catalogue.head()

,artist_name,first_release_date,primary_type,is_remix,is_compilation,is_live,is_dj_mix
0,Armin van Buuren,1996-01-01,Single,False,False,False,False
1,Armin van Buuren,1996-01-01,Single,True,False,False,False
2,Armin van Buuren,1996-03-10,Single,False,False,False,False
3,Armin van Buuren,1999-01-01,Single,False,False,False,False
4,Armin van Buuren,1999-01-01,EP,False,False,False,False


In [6]:
CUTOFF_MONTH = 9
CUTOFF_DAY = 29

catalogue["release_year"] = (
    catalogue["first_release_date"].dt.year
)

catalogue["within_comparable_period"] = (
    (catalogue["first_release_date"].dt.month < CUTOFF_MONTH)
    |
    (
        (catalogue["first_release_date"].dt.month == CUTOFF_MONTH)
        &
        (catalogue["first_release_date"].dt.day <= CUTOFF_DAY)
    )
)

comparable = (
    catalogue[
        catalogue["within_comparable_period"]
    ]
    .groupby(
        ["artist_name", "release_year"]
    )
    .size()
    .reset_index(name="comparable_releases")
)

comparable.tail(12)

,artist_name,release_year,comparable_releases
48,Tiësto,2015,11
49,Tiësto,2016,10
50,Tiësto,2017,8
51,Tiësto,2018,14
52,Tiësto,2019,14
53,Tiësto,2020,13
54,Tiësto,2021,7
55,Tiësto,2022,10
56,Tiësto,2023,14
57,Tiësto,2024,15


In [7]:
recent_baseline = (
    comparable[
        comparable["release_year"].between(
            2021,
            2025
        )
    ]
    .groupby("artist_name")
    .agg(
        avg_comparable_releases_5y=(
            "comparable_releases",
            "mean"
        ),
        median_comparable_releases_5y=(
            "comparable_releases",
            "median"
        )
    )
    .reset_index()
)

current_2026 = (
    comparable[
        comparable["release_year"] == 2026
    ][
        [
            "artist_name",
            "comparable_releases"
        ]
    ]
    .rename(
        columns={
            "comparable_releases":
            "releases_2026_ytd"
        }
    )
)

momentum_context = current_2026.merge(
    recent_baseline,
    on="artist_name",
    how="left"
)

momentum_context["vs_5y_avg_pct"] = (
    (
        momentum_context["releases_2026_ytd"]
        / momentum_context["avg_comparable_releases_5y"]
    )
    - 1
) * 100

momentum_context.round(1)

,artist_name,releases_2026_ytd,avg_comparable_releases_5y,median_comparable_releases_5y,vs_5y_avg_pct
0,Armin van Buuren,28,22.4,22.0,25.0
1,Tiësto,3,12.2,14.0,-75.4


In [8]:
catalogue["is_core"] = (
    ~catalogue["is_remix"]
    & ~catalogue["is_compilation"]
    & ~catalogue["is_live"]
    & ~catalogue["is_dj_mix"]
)

comparable_detail = (
    catalogue[
        catalogue["within_comparable_period"]
    ]
    .groupby(
        ["artist_name", "release_year"]
    )
    .agg(
        total_releases=(
            "first_release_date",
            "size"
        ),
        core_releases=(
            "is_core",
            "sum"
        )
    )
    .reset_index()
)

comparable_detail["broader_releases"] = (
    comparable_detail["total_releases"]
    - comparable_detail["core_releases"]
)

comparable_detail["core_share_pct"] = (
    comparable_detail["core_releases"]
    / comparable_detail["total_releases"]
    * 100
)

comparable_detail[
    comparable_detail["release_year"] >= 2021
].round(1)

,artist_name,release_year,total_releases,core_releases,broader_releases,core_share_pct
23,Armin van Buuren,2021,24,16,8,66.7
24,Armin van Buuren,2022,22,15,7,68.2
25,Armin van Buuren,2023,19,14,5,73.7
26,Armin van Buuren,2024,22,19,3,86.4
27,Armin van Buuren,2025,25,19,6,76.0
28,Armin van Buuren,2026,28,18,10,64.3
54,Tiësto,2021,7,2,5,28.6
55,Tiësto,2022,10,4,6,40.0
56,Tiësto,2023,14,6,8,42.9
57,Tiësto,2024,15,13,2,86.7


In [9]:
audience = con.execute("""
    SELECT
        artist_name,
        snapshot_date,
        listeners,
        playcount,
        playcount_per_listener,
        previous_listeners,
        listener_change,
        listener_growth_pct,
        previous_playcount,
        playcount_change,
        playcount_growth_pct
    FROM gold_lastfm_audience_metrics
    ORDER BY artist_name, snapshot_date
""").fetchdf()

audience

,artist_name,snapshot_date,listeners,playcount,playcount_per_listener,previous_listeners,listener_change,listener_growth_pct,previous_playcount,playcount_change,playcount_growth_pct
0,Armin van Buuren,2026-09-27,1787610,62506007,34.97,<NA>,<NA>,NaN,<NA>,<NA>,NaN
1,Armin van Buuren,2026-09-29,1787889,62513382,34.96,1787610,279,0.02,62506007,7375,0.01
2,Tiësto,2026-09-29,2823523,71266760,25.24,<NA>,<NA>,NaN,<NA>,<NA>,NaN


In [10]:
latest_audience = (
    audience
    .sort_values("snapshot_date")
    .groupby("artist_name")
    .tail(1)
    [
        [
            "artist_name",
            "snapshot_date",
            "listeners",
            "playcount",
            "playcount_per_listener"
        ]
    ]
    .reset_index(drop=True)
)

latest_audience

,artist_name,snapshot_date,listeners,playcount,playcount_per_listener
0,Armin van Buuren,2026-09-29,1787889,62513382,34.96
1,Tiësto,2026-09-29,2823523,71266760,25.24
